# 18 — Do same-problem neighbors contribute to lower correct-prefix ID?

This English Colab notebook tests a targeted hypothesis suggested by notebook 17. It uses existing Phi-2 representations and the original FINAL 13 common_clean cohort. **No generation, extraction, probe training, or rerun of previous notebooks is required.**

## Predictions stated before this experiment

For `pre_last`, excluding same-problem neighbors should increase correct-class ID more than a matched random exclusion does. If the correct-class increase exceeds the incorrect-class increase, the correctness gap should shrink relative to the random reference.

Primary paired quantities:

- Correct effect = ID_C(same-problem exclusion) − ID_C(random exclusion): predicted **positive**.
- Gap effect = DeltaID(same-problem exclusion) − DeltaID(random exclusion): predicted **negative**.

Primary settings are k=10 and k=20, with the existing L24–L31 window. Report them separately, not whichever is more favorable. The full fixed k grid (5,8,10,15,20,30,40), all 32 layers, incorrect-class effects and `finals` comparison are retained as diagnostics. This is exploratory: the hypothesis was informed by earlier results on these data, not preregistered external confirmation.

## What is held fixed

Each draw contains the exact original 397 correct and 397 incorrect query points from 129 problems. The same query points and their weights are used in every condition. Original matched plans, cohort, float64 row L2 normalization and mean-of-local-inverses Levina–Bickel formula are preserved. No coordinate standardization is used. k excludes the query itself; candidates always belong to its correctness class.

## Three conditions

1. **Original:** every other point in the class is eligible.
2. **Exclude same problem:** forbid every other attempt from the query's problem.
3. **Random exclusion:** forbid exactly the same number of uniformly selected nonself candidates for that query. Selection is identity-blind and can include same-problem points. Twenty random repetitions in FINAL (five in QUICK) are averaged *inside each matched draw*.

For a query with four other same-problem candidates, conditions 2 and 3 each retain 392 of the original 396 candidates. Both still use exactly k neighbors. Queries with zero same-problem candidates are identical in all three conditions. Random exclusion schedules are fixed before distances are examined and shared across views, layers and k.

**Interpretation:** restricting candidate neighborhoods changes the measurement procedure. The restricted results are conditional geometric diagnostics, not automatically estimates of the original unconditional manifold ID. This is an intervention on the estimator's neighbor selection, not on the language model.

## Distance diagnostics

Inspect nearest same-problem and nearest different-problem distances, their ratios to the original kth-neighbor distance, and same-problem occupancy among the first k ranks. For nearest-same comparisons, only queries with at least one same-problem candidate are eligible; denominators are reported. Candidate pool sizes differ, so these descriptive comparisons alone do not establish an effect of problem identity.

## Colab steps

1. Use a GPU runtime if available. Leave QUICK=True and RUN_DIAGNOSTICS=True in cell 1.
2. Run cells 1–7 in order and mount Drive. The source FINAL 13 folder is located by its existing manifest hash. Set SOURCE13 manually if needed; its extracted aligned review bundle also works.
3. Send the QUICK review_bundle.zip for inspection. QUICK checks L0/L24/L31 with three draws and five random repetitions; it does not produce a late-layer average.
4. Then set QUICK=False and run cells 1–7 for the 32-layer, 120-draw, 20-random-repeat analysis. Keep configuration fixed once results are inspected.

Before each layer is analysed, all 120 original draws reproduce the saved k=10/20 baseline. A mismatch or numerical failure stops execution. Checkpoints are saved per view/layer. Following interruption, run in order with the same settings; completed layers are validated and skipped. The random schedule is also checkpointed. Changed configuration/software/device produces a separate result directory.

No new LOPO or permutation inference is added here. Percentiles across matched draws are not confidence intervals; random repetitions are Monte Carlo computation, not extra research samples.

In [ ]:
# 1 — Configuration. Run all cells in order. QUICK is an execution test, not final evidence.
import json, hashlib, platform, gc, shutil, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display
try:
    from google.colab import drive
except ImportError:
    pass
else:
    drive.mount('/content/drive')

BASE = Path('/content/drive/MyDrive/msc_thesis')
SOURCE13 = None  # Optional: exact FINAL 13 directory, or extracted aligned_final_review_bundle.
QUICK = True
RUN_DIAGNOSTICS = True  # False runs the input/plan audit only; cells 5–7 then stop explicitly.
K_VALUES = [5, 8, 10, 15, 20, 30, 40]
VIEWS = ['finals', 'pre_last']
CAPS = [10]  # Only the original full matched sample; no cap sensitivity in this experiment.
LAYERS = [0, 24, 31] if QUICK else list(range(32))
DRAWS = 3 if QUICK else 120
RANDOM_REPEATS = 5 if QUICK else 20
EXCLUSION_SEED = 1818
DETAIL_DRAWS = [0]  # Fixed in advance. Full neighbor/local records for these draws only.
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
EXPECTED_SOURCE_SHA = '2bb21797cc39443a066b47a9afab2bece98d99e08879562d05ca0d0c7df08e74'
ORIGINAL = BASE/'data/representations/phi2/phi2_repr_paired_v1.pt'
PREFIX = BASE/'data/representations/phi2/phi2_preanswer_paired_v1.pt'
CONFIG = dict(version='neighbor_exclusion_v1', random_repeats=RANDOM_REPEATS, exclusion_seed=EXCLUSION_SEED, quick=QUICK, k_values=K_VALUES,
    views=VIEWS, layers=LAYERS, draws=DRAWS, caps=CAPS, detail_draws=DETAIL_DRAWS,
    normalize=True, dtype='float64', distance='direct cdist, no matmul',
    estimator='mean of local inverses, k-1 denominator; identical to notebook 13',
    tie_rule='stable original plan order', reference_reps=120,
    late_window=list(range(24,32)), source_manifest_sha256=EXPECTED_SOURCE_SHA,
    implementation_sha256='a531d65665727716a3605013bec9af89a21ff99bfc4ab7e1d668fc7196b673a7')

def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''): h.update(block)
    return h.hexdigest()

def bool_values(values):
    a = pd.Series(values).astype(str).str.lower()
    if not a.isin(['true','false','0','1']).all(): raise ValueError('Invalid Boolean values')
    return a.isin(['true','1']).to_numpy()

print('Mode:', 'QUICK' if QUICK else 'FINAL', '| device:', DEVICE)
print('Layers:', LAYERS, '| draws:', DRAWS, '| fixed k grid:', K_VALUES)


In [ ]:
# 2 — Numerical and exclusion functions.
"""Neighborhood diagnostics. NumPy only; no fitting, jitter, clipping or filtering."""
import hashlib
import numpy as np


def normalize_rows(x):
    x = np.asarray(x, np.float64)
    if x.ndim != 2 or not np.isfinite(x).all():
        raise ValueError('Expected finite [N,D] vectors')
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    if (norms <= 1e-12).any():
        raise FloatingPointError('Zero/near-zero input norm')
    return x / norms


def local_lb(distances, k):
    a = np.asarray(distances, np.float64)[..., :k]
    if k < 2 or a.shape[-1] != k or not np.isfinite(a).all() or (a <= 1e-12).any():
        raise FloatingPointError(f'Invalid neighbors at k={k}; no points removed')
    denominator = np.log(a[..., -1:] / a[..., :-1]).mean(axis=-1)
    if not np.isfinite(denominator).all() or (denominator <= 1e-12).any():
        raise FloatingPointError(f'Degenerate LB denominator at k={k}; no clipping')
    local = 1.0 / denominator
    if not np.isfinite(local).all():
        raise FloatingPointError('Nonfinite local ID')
    return local, denominator


def validate_plans(plans, pids, labels, cap=10):
    plans = np.asarray(plans)
    if plans.ndim != 3 or plans.shape[1] != 2 or plans.dtype.kind not in 'iu':
        raise ValueError('Plans must be integer [draw,2,N]')
    if plans.min() < 0 or plans.max() >= len(pids):
        raise ValueError('Plan index out of cohort bounds')
    for C, I in plans:
        if len(np.unique(np.r_[C, I])) != 2 * len(C):
            raise ValueError('Repeated attempt in matched draw')
        if not labels[C].all() or labels[I].any():
            raise ValueError('Wrong class in matched plan')
        if not np.array_equal(pids[C], pids[I]):
            raise ValueError('Unmatched problem identities/order')
        ids, counts = np.unique(pids[C], return_counts=True)
        expected = np.asarray([min(cap, np.sum((pids == p) & labels), np.sum((pids == p) & ~labels)) for p in ids])
        if not np.array_equal(ids, np.unique(pids)) or not np.array_equal(counts, expected):
            raise ValueError('Unexpected problem support or counts')


def regenerated_plans(pids, labels, cap, reps, seed):
    # Reproduce notebook 13 RNG calls and original attempt order exactly.
    rng = np.random.default_rng(seed)
    result = []
    for _ in range(reps):
        C, I = [], []
        for pid in np.unique(pids):
            c = np.flatnonzero((pids == pid) & labels)
            i = np.flatnonzero((pids == pid) & ~labels)
            n = min(cap, len(c), len(i))
            C.extend(rng.choice(c, n, replace=False))
            I.extend(rng.choice(i, n, replace=False))
        result.append([C, I])
    return np.asarray(result, dtype=np.int64)


def nested_plans(full, pids, cap):
    # Random order comes from the original draw, shared across all representations/layers/k.
    result = []
    for C, I in full:
        keep = np.zeros(len(C), bool)
        for pid in np.unique(pids[C]):
            keep[np.flatnonzero(pids[C] == pid)[:cap]] = True
        result.append([C[keep], I[keep]])
    return np.asarray(result, dtype=np.int64)


def neighbors(D, indices, max_k):
    indices = np.asarray(indices, int)
    if len(indices) <= max_k + 1:
        raise ValueError('Need N > max_k+1 to audit boundary ties')
    sub = np.asarray(D[np.ix_(indices, indices)], np.float64).copy()
    np.fill_diagonal(sub, np.inf)
    # Stable sort fixes equal-distance ties by original plan position, never label or ID estimate.
    pos = np.argsort(sub, axis=1, kind='stable')[:, :max_k + 1]
    d = np.take_along_axis(sub, pos, axis=1)
    if not np.isfinite(d).all() or (d <= 1e-12).any():
        raise FloatingPointError('Nonpositive/invalid neighbor distance; inspect source, do not discard')
    return indices[pos], d


def diagnose_class(D, indices, pids, k_values):
    indices = np.asarray(indices, int)
    nn, d = neighbors(D, indices, max(k_values))
    same = pids[nn] == pids[indices, None]
    n = len(indices)
    counts = {p: int(np.sum(pids[indices] == p)) for p in np.unique(pids[indices])}
    expected = np.asarray([(counts[p] - 1) / (n - 1) for p in pids[indices]])
    all_local, all_den, rows = [], [], []
    tail_n = max(1, int(np.ceil(.05 * n)))
    for k in k_values:
        local, den = local_lb(d, k)
        all_local.append(local); all_den.append(den)
        own = same[:, :k].mean(axis=1)
        q = np.quantile(local, [.05, .25, .5, .75, .9, .95, .99])
        top = np.sort(local)[-tail_n:]
        rows.append(dict(k=int(k), n=int(n), id_mean=float(local.mean()),
            local_min=float(local.min()), local_p05=q[0], local_p25=q[1], local_median=q[2],
            local_p75=q[3], local_p90=q[4], local_p95=q[5], local_p99=q[6], local_max=float(local.max()),
            top5_share=float(top.sum()/local.sum()), top5_n=tail_n,
            denominator_min=float(den.min()), denominator_p01=float(np.quantile(den,.01)),
            nearest_min=float(d[:,0].min()), radius_mean=float(d[:,k-1].mean()),
            radius_median=float(np.median(d[:,k-1])), radius_p90=float(np.quantile(d[:,k-1],.9)),
            own_fraction=float(own.mean()), own_expected=float(expected.mean()),
            own_excess=float((own-expected).mean()),
            adjacent_tie_fraction=float((np.diff(d[:,:k],axis=1)==0).mean()),
            boundary_tie_fraction=float((d[:,k-1]==d[:,k]).mean())))
    rank_rows = [dict(rank=j+1, distance_mean=float(d[:,j].mean()),
        distance_median=float(np.median(d[:,j])), own_fraction=float(same[:,j].mean()),
        own_expected=float(expected.mean())) for j in range(max(k_values))]
    local = np.stack(all_local, axis=1)
    shift = local[:,k_values.index(20)]-local[:,k_values.index(10)]
    top_abs = np.argsort(np.abs(shift), kind='stable')[-tail_n:]
    total_abs = np.abs(shift).sum()
    shift_row = dict(n=n, shift_mean=float(shift.mean()), shift_median=float(np.median(shift)),
        shift_p05=float(np.quantile(shift,.05)), shift_p95=float(np.quantile(shift,.95)),
        fraction_negative=float((shift<0).mean()), fraction_positive=float((shift>0).mean()),
        positive_contribution=float(np.maximum(shift,0).sum()/n),
        negative_contribution=float(np.minimum(shift,0).sum()/n),
        top_abs5_contribution=float(shift[top_abs].sum()/n),
        remainder_contribution=float((shift.sum()-shift[top_abs].sum())/n),
        top_abs5_share=float(np.abs(shift[top_abs]).sum()/total_abs) if total_abs else 0.0,
        own_first10=float(same[:,:10].mean()), own_next10=float(same[:,10:20].mean()),
        radius20_over10_median=float(np.median(d[:,19]/d[:,9])))
    detail = dict(query_indices=indices, neighbor_indices=nn, distances=d,
                  local_id=local, denominators=np.stack(all_den,axis=1), own_problem=same)
    return rows, rank_rows, shift_row, detail


def baseline_ids(D, plans, k_values=(10,20)):
    result = np.empty((len(plans),2,len(k_values)))
    for r, plan in enumerate(plans):
        for side, ids in enumerate(plan):
            # Mirrors notebook 13: argpartition then sorted distances.
            sub = np.asarray(D[np.ix_(ids, ids)],np.float64).copy()
            np.fill_diagonal(sub,np.inf)
            d = np.partition(sub,max(k_values)-1,axis=1)[:,:max(k_values)]
            d.sort(axis=1)
            for j,k in enumerate(k_values): result[r,side,j] = local_lb(d,k)[0].mean()
    return result


def atomic_npz(path, **arrays):
    temp = path.with_suffix('.tmp')
    with temp.open('wb') as f: np.savez_compressed(f,**arrays)
    temp.replace(path)


def read_signed_npz(path, signature):
    with np.load(path,allow_pickle=False) as z:
        if str(z['signature'].item()) != signature:
            raise ValueError(f'Stale checkpoint: {path}')
        return {k:z[k].copy() for k in z.files if k!='signature'}

"""Query-specific neighbor exclusions; all query points and weights are retained."""
def exclusion_schedule(plan, pids, repeats, seed):
    n=len(plan); groups=pids[plan]
    counts=(groups[:,None]==groups[None,:]).sum(axis=1)-1
    max_m=int(counts.max())
    if max_m==0:return np.empty((repeats,n,0),dtype=np.int16)
    rng=np.random.default_rng(seed)
    # Uniform random priorities over all nonself candidates, independent per query/repeat.
    keys=rng.random((repeats,n,n)); idx=np.arange(n);keys[:,idx,idx]=np.inf
    chosen=np.argpartition(keys,max_m-1,axis=2)[:,:,:max_m]
    vals=np.take_along_axis(keys,chosen,axis=2)
    chosen=np.take_along_axis(chosen,np.argsort(vals,axis=2),axis=2)
    chosen=np.where(np.arange(max_m)[None,None,:]<counts[None,:,None],chosen,-1)
    return chosen.astype(np.int16)


def excluded_statistics(D, ids, pids, ks, schedule):
    ids=np.asarray(ids,int);n=len(ids);groups=pids[ids]
    same=(groups[:,None]==groups[None,:]);np.fill_diagonal(same,False)
    m=same.sum(axis=1);maxm=int(m.max());needed=max(ks)+maxm
    if n-1-maxm<max(ks):raise ValueError('Insufficient eligible candidates')
    sub=D[np.ix_(ids,ids)].copy();np.fill_diagonal(sub,np.inf)
    order=np.argsort(sub,axis=1,kind='stable')[:,:needed]
    distances=np.take_along_axis(sub,order,axis=1)
    if not np.isfinite(distances).all() or (distances<=1e-12).any():raise FloatingPointError('Invalid distances')
    def calc(a):return np.stack([local_lb(a,k)[0] for k in ks],axis=1)
    original=calc(distances)
    forbidden=np.take_along_axis(same,order,axis=1)
    cross=np.sort(np.where(forbidden,np.inf,distances),axis=1)[:,:max(ks)]
    excluded=calc(cross)
    random_local=[];random_removed_near=[]
    for chosen in schedule:
        forbidden_random=(order[:,:,None]==chosen[:,None,:]).any(axis=2)
        allowed=np.sort(np.where(forbidden_random,np.inf,distances),axis=1)[:,:max(ks)]
        random_local.append(calc(allowed))
        random_removed_near.append([forbidden_random[:,:k].sum(axis=1).mean() for k in ks])
    random_local=np.stack(random_local)
    # Distance diagnostics conditional on queries with >=1 same-problem candidate.
    valid=m>0
    same_nearest=np.where(same,sub,np.inf).min(axis=1)
    other_nearest=np.where(~same,sub,np.inf).min(axis=1)
    diagnostics=[]
    for j,k in enumerate(ks):
        radius=distances[:,k-1]
        diagnostics.append(dict(k=k,queries=n,queries_with_same=int(valid.sum()),
            same_fraction=float(forbidden[:,:k].mean()),
            same_expected=float(np.mean(m/(n-1))),
            same_nearest_mean=float(same_nearest[valid].mean()) if valid.any() else None,
            other_nearest_mean=float(other_nearest[valid].mean()) if valid.any() else None,
            same_nearest_over_Tk_median=float(np.median(same_nearest[valid]/radius[valid])) if valid.any() else None,
            other_nearest_over_Tk_median=float(np.median(other_nearest[valid]/radius[valid])) if valid.any() else None,
            fraction_same_closer=float((same_nearest[valid]<other_nearest[valid]).mean()) if valid.any() else None,
            original_radius_mean=float(radius.mean()), excluded_radius_mean=float(cross[:,k-1].mean()),
            random_removed_topk_mean=float(np.asarray(random_removed_near)[:,j].mean())))
    return original,excluded,random_local,diagnostics,dict(query_indices=ids,neighbor_indices=ids[order],
        neighbor_distances=distances,same_problem_mask=forbidden,same_candidate_count=m,
        nearest_same=same_nearest,nearest_other=other_nearest)


In [ ]:
# 3 — Verify original cohort, cache provenance, and all 120 original plans.
if SOURCE13 is None:
    candidates = sorted((BASE/'results/aligned_four_views_phi2_v1/final').glob('*/manifest.json'))
    matches = [p.parent for p in candidates if digest(p)==EXPECTED_SOURCE_SHA]
    if len(matches)!=1:
        raise ValueError('Set SOURCE13 to the original FINAL 13 folder (or its extracted review bundle). '
                         'Candidates: '+str([str(p.parent) for p in candidates]))
    SOURCE13 = matches[0]
SOURCE13 = Path(SOURCE13)
if digest(SOURCE13/'manifest.json') != EXPECTED_SOURCE_SHA:
    raise ValueError('Source manifest differs from the aligned analysis being diagnosed')
SOURCE_MANIFEST = json.loads((SOURCE13/'manifest.json').read_text())
sc = SOURCE_MANIFEST['config']
assert sc['quick'] is False and sc['pooled_reps']==120 and sc['cap']==10 and sc['normalize'] is True
assert sc['k_values']==[10,20] and sc['layers']==list(range(32))
for path in [ORIGINAL,PREFIX]:
    expected = [v for k,v in SOURCE_MANIFEST['inputs'].items() if Path(k).name==path.name]
    if len(expected)!=1 or digest(path)!=expected[0]:
        raise ValueError(f'Representation cache differs from FINAL 13: {path}')

all_meta = pd.read_csv(SOURCE13/'cohort_membership.csv')
assert not all_meta.duplicated(['problem_id','attempt_id']).any()
all_keys = all_meta[['problem_id','attempt_id']].to_numpy(int)
assert np.array_equal(all_keys,np.asarray(SOURCE_MANIFEST['common_keys'],int))
clean_indices = np.asarray(SOURCE_MANIFEST['clean_indices'],int)
assert np.array_equal(clean_indices,np.flatnonzero(~bool_values(all_meta.excluded)))
META = all_meta.iloc[clean_indices].reset_index(drop=True).copy()
META['correct'] = bool_values(META.correct)
KEYS = META[['problem_id','attempt_id']].to_numpy(int)
PIDS = META.problem_id.to_numpy(int)
LABELS = META.correct.to_numpy(bool)
with np.load(SOURCE13/'common_clean_plans.npz',allow_pickle=False) as z:
    FULL_PLANS = z['pooled'].copy()
    assert np.array_equal(z['cohort_keys'],KEYS)
validate_plans(FULL_PLANS,PIDS,LABELS,cap=10)
assert len(FULL_PLANS)==120
expected = regenerated_plans(PIDS,LABELS,10,120,sc['matching_seed'])
assert np.array_equal(FULL_PLANS,expected), 'Saved plans do not match original sampling protocol'
del expected
REFERENCE = pd.read_csv(SOURCE13/'aligned_id_summary.csv')
REFERENCE = REFERENCE[REFERENCE.cohort=='common_clean'].copy()
assert not REFERENCE.duplicated(['representation','layer','k']).any()
for v in VIEWS:
    q = REFERENCE[REFERENCE.representation==v]
    assert len(q)==64 and set(q.k)=={10,20} and set(q.layer)==set(range(32))
    assert (q.matched_per_class==FULL_PLANS.shape[-1]).all() and (q.reps_pooled==120).all()
assert len(META)==2541 and len(np.unique(PIDS))==129 and LABELS.sum()==433
print('Verified source:',SOURCE13)
print('Cohort:',len(META),'attempts,',len(np.unique(PIDS)),'problems')
print('Original plans:',FULL_PLANS.shape,'| preserved exactly')


In [ ]:
# 4 — Freeze nested plans and output signature. No ID calculation yet.
PLANS = {cap:nested_plans(FULL_PLANS[:DRAWS],PIDS,cap) for cap in CAPS}
assert np.array_equal(PLANS[10],FULL_PLANS[:DRAWS])
plan_rows = []
for cap,plans in PLANS.items():
    validate_plans(plans,PIDS,LABELS,cap)
    assert plans.shape[-1] > max(K_VALUES)+1
    for r,(C,I) in enumerate(plans):
        for side,ids in enumerate([C,I]):
            assert set(ids)<=set(FULL_PLANS[r,side])
        plan_rows.append(dict(cap=cap,draw=r,n_per_class=len(C),n_problems=len(np.unique(PIDS[C]))))
PLAN_SUMMARY = pd.DataFrame(plan_rows)
input_hashes = {str(p):digest(p) for p in [SOURCE13/'manifest.json',SOURCE13/'cohort_membership.csv',
    SOURCE13/'common_clean_plans.npz',SOURCE13/'aligned_id_summary.csv']}
# Cache hashes were verified above; do not re-read multi-GB representations.
for path in [ORIGINAL,PREFIX]:
    input_hashes[str(path)] = next(v for k,v in SOURCE_MANIFEST['inputs'].items() if Path(k).name==path.name)
MANIFEST = dict(config=CONFIG, inputs=input_hashes,
    versions=dict(python=platform.python_version(),numpy=np.__version__,pandas=pd.__version__,torch=torch.__version__),
    device=DEVICE, source13=str(SOURCE13), n_attempts=len(META), n_problems=len(np.unique(PIDS)),
    notes=['Exploratory follow-up motivated by existing results; not independent confirmation.',
           'All original query points/weights are retained; only neighbor eligibility changes.',
           'Detailed neighbors saved only for configured detail_draws; summaries use every draw.',
           'No permutation p-values or new LOPO inference in this diagnostic notebook.'])
SIGNATURE = hashlib.sha256(json.dumps(MANIFEST,sort_keys=True).encode()).hexdigest()
OUT = BASE/'results/neighbor_exclusion_phi2_v1'/('quick' if QUICK else 'final')/SIGNATURE[:12]
OUT.mkdir(parents=True,exist_ok=True)
(OUT/'checkpoints').mkdir(exist_ok=True)
(OUT/'manifest.json').write_text(json.dumps(MANIFEST,indent=2))
META.to_csv(OUT/'cohort_keys.csv',index=False)
PLAN_SUMMARY.to_csv(OUT/'plan_summary.csv',index=False)
atomic_npz(OUT/'matched_plans.npz',signature=np.asarray(SIGNATURE),cohort_keys=KEYS,
           **{f'cap_{cap}':p for cap,p in PLANS.items()})
print('Results:',OUT)
display(PLAN_SUMMARY.groupby('cap')[['n_per_class','n_problems']].first())
print('Original points and per-problem weights retained in all conditions.')

# Freeze query-specific random exclusions before examining distances. These do not use vectors.
schedule_path=OUT/'random_exclusions.npz'
if schedule_path.exists():
    SCHEDULE=read_signed_npz(schedule_path,SIGNATURE)['schedule']
else:
    schedules=[]
    for r,plan in enumerate(tqdm(FULL_PLANS[:DRAWS],desc='Freeze random candidate exclusions')):
        schedules.append([exclusion_schedule(ids,PIDS,RANDOM_REPEATS,EXCLUSION_SEED+2*r+side)
                          for side,ids in enumerate(plan)])
    SCHEDULE=np.asarray(schedules,dtype=np.int16)
    atomic_npz(schedule_path,signature=np.asarray(SIGNATURE),schedule=SCHEDULE)
assert SCHEDULE.shape[:4]==(DRAWS,2,RANDOM_REPEATS,FULL_PLANS.shape[-1])
for r,plan in enumerate(FULL_PLANS[:DRAWS]):
    for side,ids in enumerate(plan):
        counts=(PIDS[ids,None]==PIDS[ids][None,:]).sum(axis=1)-1
        s=SCHEDULE[r,side]
        assert np.array_equal((s>=0).sum(axis=2),np.broadcast_to(counts,s.shape[:2]))
        assert not (s==np.arange(len(ids))[None,:,None]).any()
        assert ((s==-1)|((s>=0)&(s<len(ids)))).all()
        for rep in s:
            for chosen in rep:
                a=chosen[chosen>=0];assert len(a)==len(np.unique(a))
print('Frozen random repeats per draw:',RANDOM_REPEATS)


In [ ]:
# 5 — Reproduce original results, then run the three fixed conditions.
def checkpoint_path(view,layer):
    return OUT/'checkpoints'/f'{view}__L{layer:02d}.npz'

@torch.no_grad()
def distance_matrix(raw):
    x=torch.from_numpy(normalize_rows(raw)).to(DEVICE)
    D=np.empty((len(x),len(x)),np.float64)
    for start in range(0,len(x),256):
        D[start:start+256]=torch.cdist(x[start:start+256],x,p=2,
            compute_mode='donot_use_mm_for_euclid_dist').cpu().numpy()
    np.fill_diagonal(D,np.inf)
    off=~np.eye(len(D),dtype=bool)
    if ((D[off]<=1e-12)|~np.isfinite(D[off])).any():
        raise FloatingPointError('Invalid/zero distances; no new filtering, jitter or clipping is allowed')
    if not np.allclose(D,D.T,rtol=1e-12,atol=1e-12):raise FloatingPointError('Asymmetric distances')
    return D

if not RUN_DIAGNOSTICS:raise RuntimeError('Audit complete. Set RUN_DIAGNOSTICS=True and run cells 5–7.')
for view in VIEWS:
    pending=[]
    for layer in LAYERS:
        path=checkpoint_path(view,layer)
        if path.exists():read_signed_npz(path,SIGNATURE)
        else:pending.append(layer)
    if not pending:
        print(view,': checkpoints complete');continue
    cache=torch.load(ORIGINAL if view=='finals' else PREFIX,map_location='cpu',weights_only=False)
    lookup={(int(r['problem_id']),int(r['attempt_id'])):r for r in cache}
    assert len(lookup)==len(cache)
    records=[lookup[tuple(key)] for key in KEYS]
    for j,r in enumerate(records):
        assert bool_values([r['correct']])[0]==LABELS[j]
        assert tuple(r[view].shape)==(32,2560)
    tensors=[r[view] for r in records];del records,lookup,cache;gc.collect()
    for layer in tqdm(pending,desc=view):
        D=distance_matrix(np.stack([t[layer].double().numpy() for t in tensors]))
        baseline=baseline_ids(D,FULL_PLANS)
        checks=[]
        for j,k in enumerate([10,20]):
            ref=REFERENCE[(REFERENCE.representation==view)&(REFERENCE.layer==layer)&(REFERENCE.k==k)].iloc[0]
            vals=[baseline[:,0,j].mean(),baseline[:,1,j].mean(),(baseline[:,1,j]-baseline[:,0,j]).mean()]
            for metric,value in zip(['pooled_id_correct','pooled_id_incorrect','pooled_delta_id'],vals):
                checks.append(dict(representation=view,layer=layer,k=k,metric=metric,reference=float(ref[metric]),
                    recomputed=float(value),difference=float(value-ref[metric]),
                    passed=bool(np.isclose(value,float(ref[metric]),rtol=1e-6,atol=1e-8))))
        if not all(c['passed'] for c in checks):
            pd.DataFrame(checks).to_csv(OUT/f'reproduction_failure_{view}_{layer}.csv',index=False)
            raise RuntimeError('Original reproduction failed; stop before new interpretation')
        observed=np.empty((DRAWS,2,3,len(K_VALUES))) # draw, class, original/same-excluded/random-mean, k
        randomized=np.empty((DRAWS,2,RANDOM_REPEATS,len(K_VALUES)))
        diag=[];local_changes=[];details={}
        for r,plan in enumerate(tqdm(FULL_PLANS[:DRAWS],desc=f'L{layer:02d} draws',leave=False)):
            for side,ids in enumerate(plan):
                try:
                    original,excluded,random_local,diagnostics,detail=excluded_statistics(
                        D,ids,PIDS,K_VALUES,SCHEDULE[r,side])
                except (FloatingPointError,ValueError) as exc:
                    raise RuntimeError(f'{view} L{layer} draw={r} class={side}: {exc}') from exc
                random_mean=random_local.mean(axis=0)
                observed[r,side]=np.stack([original.mean(axis=0),excluded.mean(axis=0),random_mean.mean(axis=0)])
                randomized[r,side]=random_local.mean(axis=1)
                for k,j in [(10,0),(20,1)]:assert np.isclose(observed[r,side,0,K_VALUES.index(k)],baseline[r,side,j],rtol=1e-12,atol=1e-12)
                zero=detail['same_candidate_count']==0
                assert np.allclose(original[zero],excluded[zero])
                assert np.allclose(random_local[:,zero],original[None,zero])
                context=dict(representation=view,layer=layer,draw=r,group=['C','I'][side])
                diag.extend([{**context,**a} for a in diagnostics])
                for j,k in enumerate(K_VALUES):
                    shift=excluded[:,j]-random_mean[:,j]
                    n_tail=max(1,int(np.ceil(.05*len(shift))))
                    top=np.argsort(np.abs(shift),kind='stable')[-n_tail:]
                    local_changes.append(dict(**context,k=k,mean=float(shift.mean()),median=float(np.median(shift)),
                        p05=float(np.quantile(shift,.05)),p95=float(np.quantile(shift,.95)),
                        fraction_positive=float((shift>0).mean()),top_abs5_contribution=float(shift[top].sum()/len(shift)),
                        remainder_contribution=float((shift.sum()-shift[top].sum())/len(shift))))
                if r in DETAIL_DRAWS:
                    stem=f'draw{r}_{context["group"]}_'
                    for name,a in detail.items():details[stem+name]=a
                    details[stem+'original_local']=original;details[stem+'excluded_local']=excluded
                    details[stem+'random_mean_local']=random_mean
        atomic_npz(checkpoint_path(view,layer),signature=np.asarray(SIGNATURE),observed=observed,randomized=randomized,
            diagnostics_json=np.asarray(pd.DataFrame(diag).to_json(orient='records',double_precision=15)),
            local_changes_json=np.asarray(pd.DataFrame(local_changes).to_json(orient='records',double_precision=15)),
            reproduction_json=np.asarray(pd.DataFrame(checks).to_json(orient='records',double_precision=15)),**details)
        del D,details,diag,local_changes;gc.collect()
    del tensors;gc.collect()
print('All requested layers complete; source reproduction passed.')


In [ ]:
# 6 — Paired effects. Average random repeats inside each draw; never treat them as new observations.
from io import StringIO
rows=[];random_rows=[];diagnostics=[];local_changes=[];checks=[]
CONDITIONS=['original','exclude_same_problem','random_exclusion']
for view in VIEWS:
    for layer in LAYERS:
        path=checkpoint_path(view,layer)
        if not path.exists():raise RuntimeError(f'Missing checkpoint {path}; resume cell 5')
        with np.load(path,allow_pickle=False) as z:
            if str(z['signature'].item())!=SIGNATURE:raise ValueError('Stale checkpoint')
            obs=z['observed'];ran=z['randomized']
            assert obs.shape==(DRAWS,2,3,len(K_VALUES))
            assert ran.shape==(DRAWS,2,RANDOM_REPEATS,len(K_VALUES))
            assert np.isfinite(obs).all() and np.isfinite(ran).all()
            assert np.allclose(obs[:,:,2],ran.mean(axis=2))
            for r in range(DRAWS):
                for j,k in enumerate(K_VALUES):
                    for c,condition in enumerate(CONDITIONS):
                        C,I=obs[r,:,c,j]
                        rows.append(dict(representation=view,layer=layer,draw=r,k=k,condition=condition,C=C,I=I,delta=I-C))
                    for rep in range(RANDOM_REPEATS):
                        C,I=ran[r,:,rep,j]
                        random_rows.append(dict(representation=view,layer=layer,draw=r,k=k,repeat=rep,C=C,I=I,delta=I-C))
            for key,target in [('diagnostics',diagnostics),('local_changes',local_changes),('reproduction',checks)]:
                target.append(pd.read_json(StringIO(str(z[key+'_json'].item()))))
DRAW=pd.DataFrame(rows);RANDOM_DRAW=pd.DataFrame(random_rows)
DIAGNOSTICS=pd.concat(diagnostics,ignore_index=True);LOCAL=pd.concat(local_changes,ignore_index=True)
REPRO=pd.concat(checks,ignore_index=True);assert REPRO.passed.all()
assert not DRAW.duplicated(['representation','layer','draw','k','condition']).any()

def summarize(frame,keys,values):
    out=[]
    for key,g in frame.groupby(keys):
        if not isinstance(key,tuple):key=(key,)
        row=dict(zip(keys,key));row['draws']=len(g)
        for value in values:
            a=g[value].to_numpy(float)
            if not np.isfinite(a).all():raise FloatingPointError(f'Invalid metric {value}')
            row[value+'_mean']=float(a.mean());row[value+'_sampling_p025']=float(np.quantile(a,.025));row[value+'_sampling_p975']=float(np.quantile(a,.975))
        out.append(row)
    return pd.DataFrame(out)

def paired(frame,index):
    wide=frame.pivot(index=index,columns='condition',values=['C','I','delta'])
    out=[]
    for left,right in [('exclude_same_problem','random_exclusion'),('exclude_same_problem','original'),('random_exclusion','original')]:
        a=wide.index.to_frame(index=False);a['comparison']=left+' minus '+right
        for metric in ['C','I','delta']:a[metric]=(wide[(metric,left)]-wide[(metric,right)]).to_numpy()
        out.append(a)
    return pd.concat(out,ignore_index=True)

SUMMARY=summarize(DRAW,['representation','layer','k','condition'],['C','I','delta'])
EFFECT_DRAW=paired(DRAW,['representation','layer','draw','k'])
EFFECT=summarize(EFFECT_DRAW,['representation','layer','k','comparison'],['C','I','delta'])
HAVE_LATE=set(range(24,32)).issubset(LAYERS)
LATE=pd.DataFrame();LATE_EFFECT=pd.DataFrame();MC=pd.DataFrame()
if HAVE_LATE:
    late_draw=DRAW[DRAW.layer.between(24,31)].groupby(['representation','draw','k','condition'],as_index=False)[['C','I','delta']].mean()
    LATE=summarize(late_draw,['representation','k','condition'],['C','I','delta'])
    LATE_EFFECT=summarize(paired(late_draw,['representation','draw','k']),['representation','k','comparison'],['C','I','delta'])
    # Assess numerical Monte Carlo variability of the mean random reference over all draws/layers.
    # This is NOT data uncertainty; the same exclusions are paired across layers and views.
    rr=RANDOM_DRAW[RANDOM_DRAW.layer.between(24,31)].groupby(['representation','k','repeat'],as_index=False)[['C','I','delta']].mean()
    out=[]
    for (v,k),g in rr.groupby(['representation','k']):
        row=dict(representation=v,k=k,repeats=RANDOM_REPEATS)
        for metric in ['C','I','delta']:
            row[metric+'_random_repeat_sd']=g[metric].std(ddof=1)
            row[metric+'_random_mean_mc_se']=g[metric].std(ddof=1)/np.sqrt(RANDOM_REPEATS)
        out.append(row)
    MC=pd.DataFrame(out)
else:print('QUICK: no L24–L31 summary; only L0, L24 and L31 were computed.')
DS=DIAGNOSTICS.groupby(['representation','layer','group','k'],as_index=False).mean(numeric_only=True).drop(columns='draw')
LS=LOCAL.groupby(['representation','layer','group','k'],as_index=False).mean(numeric_only=True).drop(columns='draw')
exports=dict(condition_summary=SUMMARY,paired_effect_summary=EFFECT,paired_effect_draws=EFFECT_DRAW,
    condition_draws=DRAW,distance_diagnostics=DS,local_effect_summary=LS,reproduction_checks=REPRO)
if HAVE_LATE:exports.update(late_condition_summary=LATE,late_paired_effects=LATE_EFFECT,random_reference_monte_carlo=MC)
for name,frame in exports.items():frame.to_csv(OUT/(name+'.csv'),index=False)
RANDOM_DRAW.to_csv(OUT/'random_reference_by_repeat.csv',index=False)
DIAGNOSTICS.to_csv(OUT/'distance_diagnostics_by_draw.csv',index=False)
LOCAL.to_csv(OUT/'local_effect_by_draw.csv',index=False)
print('Reproduction checks:',len(REPRO),'passed')
display((LATE if HAVE_LATE else SUMMARY[SUMMARY.layer==31]).query('k in [10,20]').round(4))
display((LATE_EFFECT if HAVE_LATE else EFFECT[EFFECT.layer==31]).query('comparison=="exclude_same_problem minus random_exclusion" and k in [10,20]').round(4))
print('Predictions: C effect > 0 and delta effect < 0 for pre_last, compared with random exclusion.')
print('These are diagnostic estimator contrasts, not causal interventions on the language model.')


In [ ]:
# 7 — Figures and compact review bundle. All computed layers are retained.
FIG=OUT/'figures';FIG.mkdir(exist_ok=True)
styles={'original':'-','exclude_same_problem':'--','random_exclusion':':'}
for k in [10,20]:
    fig,axes=plt.subplots(2,3,figsize=(15,8))
    for row,view in enumerate(VIEWS):
        for col,metric in enumerate(['C','I','delta']):
            ax=axes[row,col]
            for condition in CONDITIONS:
                g=SUMMARY[(SUMMARY.representation==view)&(SUMMARY.k==k)&(SUMMARY.condition==condition)].sort_values('layer')
                ax.plot(g.layer,g[metric+'_mean'],ls=styles[condition],label=condition)
            ax.set(title=f'{view}: {metric}',xlabel='Layer',ylabel='Estimated ID' if metric!='delta' else 'Delta ID')
            ax.legend(fontsize=7)
    fig.suptitle(f'k={k} | '+('QUICK: selected layers only' if QUICK else 'All 32 layers'))
    fig.tight_layout();fig.savefig(FIG/f'conditions_k{k}.png',dpi=150);plt.show();plt.close(fig)
curves=LATE if HAVE_LATE else SUMMARY[SUMMARY.layer==31]
effects=LATE_EFFECT if HAVE_LATE else EFFECT[EFFECT.layer==31]
window='L24–L31' if HAVE_LATE else 'L31 only (QUICK)'
fig,axes=plt.subplots(2,3,figsize=(15,8))
for row,view in enumerate(VIEWS):
    for col,metric in enumerate(['C','I','delta']):
        ax=axes[row,col]
        for condition in CONDITIONS:
            g=curves[(curves.representation==view)&(curves.condition==condition)].sort_values('k')
            ax.plot(g.k,g[metric+'_mean'],ls=styles[condition],marker='o',label=condition)
        ax.set(title=f'{view}: {metric}',xlabel='k',ylabel='Estimated ID' if metric!='delta' else 'Delta ID');ax.legend(fontsize=7)
fig.suptitle(window+' | identical query points, different eligible neighbors')
fig.tight_layout();fig.savefig(FIG/'condition_k_curves.png',dpi=150);plt.show();plt.close(fig)
fig,axes=plt.subplots(1,3,figsize=(15,4))
for ax,metric in zip(axes,['C','I','delta']):
    for view in VIEWS:
        g=effects[(effects.representation==view)&(effects.comparison=='exclude_same_problem minus random_exclusion')].sort_values('k')
        ax.plot(g.k,g[metric+'_mean'],marker='o',label=view)
        ax.fill_between(g.k.to_numpy(),g[metric+'_sampling_p025'].to_numpy(),g[metric+'_sampling_p975'].to_numpy(),alpha=.15)
    ax.axhline(0,color='gray',lw=.8);ax.set(title=metric,xlabel='k',ylabel='Same-problem exclusion minus random exclusion');ax.legend()
fig.suptitle(window+' | bands: matched-draw percentiles, not confidence intervals')
fig.tight_layout();fig.savefig(FIG/'paired_effects.png',dpi=150);plt.show();plt.close(fig)

notes='''# Same-problem neighbor exclusion

See manifest.json for QUICK/FINAL configuration. The original cohort and matched plans are unchanged. Every original query is retained with its original weight. Eligible neighbors are restricted separately inside each correctness class.

Conditions: original; exclude all same-problem nonself candidates; exclude the same number of uniformly random nonself candidates for each query. Random exclusion is allowed to remove same-problem candidates too: it is identity-blind. Independent random repeats are averaged inside each draw before comparisons. Random schedules are paired across representations, layers and k, with a fixed seed. Counts are matched per query, not just on average. Queries with no other same-problem attempts are unchanged in all conditions.

Predictions fixed before new results: pre_last correct ID should increase under same-problem exclusion relative to random exclusion; the increase should exceed the incorrect-class increase, reducing Delta ID. Primary k values are 10 and 20 and the existing L24–L31 window. All other fixed k values and layers remain reported. This is exploratory, motivated by notebook 17, not an independent confirmation.

Restricted-neighbor results are conditional neighborhood diagnostics using the same formula. They are not automatically estimates of the same unconditional manifold dimension as the original cloud. No intervention is performed on model activations or generation.

Distance diagnostics use queries with at least one same-problem candidate when comparing nearest same/other distances. Their candidate pools differ in size; these comparisons are descriptive, not equal-count nearest-neighbor tests. Reported queries_with_same gives the denominator. Ratios use the original kth-neighbor distance. Random exclusion controls the number of eligible candidates, not their distance or rank; differences in rank are part of the tested hypothesis.

Percentiles are across matched draws and are not confidence intervals. Repeated draws share attempts. Monte Carlo SE, when available, describes only the numerical randomness of the random reference, not research sampling uncertainty. No new permutation test or LOPO analysis. The cap=1 sample-size confound is avoided, but the source duplicate-filtered cohort and extraction precision are inherited.

Raw original-neighbor identities/distances and local estimates for fixed draw 0 remain in signed per-layer checkpoints, excluded from this compact bundle. Random schedules for every draw are saved as random_exclusions.npz; positions index the corresponding class plan in matched_plans.npz, not the full cohort. -1 is padding. Random-reference-by-repeat and per-draw distance/local-effect CSV files remain in the full run folder.

Interpretation: predicted paired effects support a contribution of within-problem neighborhoods to the measured gap. Near-zero or opposing effects weaken that explanation under this protocol. Neither outcome proves a reasoning mechanism or establishes that text similarity causes the grouping.
'''
(OUT/'REVIEW_NOTES.md').write_text(notes)
files=[OUT/'manifest.json',OUT/'REVIEW_NOTES.md',OUT/'cohort_keys.csv',OUT/'plan_summary.csv',OUT/'matched_plans.npz']
files += [OUT/(name+'.csv') for name in exports]+sorted(FIG.glob('*.png'))
with zipfile.ZipFile(OUT/'review_bundle.zip','w',compression=zipfile.ZIP_DEFLATED) as z:
    for path in files:z.write(path,path.relative_to(OUT))
print('Review bundle:',OUT/'review_bundle.zip')
print('Keep the full run directory, including checkpoints and random_exclusions.npz.')


## How to interpret either outcome

- **Predicted effects:** supports a contribution of same-problem neighborhood structure to the measured gap. Check both classes and the distance diagnostics; do not infer an internal reasoning mechanism.
- **Same-problem and random exclusion behave similarly:** weakens the specific-neighbor explanation under this protocol. Removing candidate points alone may account for the change.
- **Opposite effects:** reconsider the proposed direction instead of changing k or filtering points after inspection.
- **Mixed effects across layers or k:** report the dependence explicitly. Late averages can conceal opposite layer-specific effects.
- **Large local effects concentrated in a few queries:** inspect the saved draw-0 examples and all-draw local-effect summaries. Do not silently replace the mean estimator with a trimmed statistic.

A positive result does not show that similar wording causes grouping. Text overlap, shared reasoning steps, answer content and extraction effects remain possible explanations. The inherited cohort filtering and stored representation precision remain limitations.

Estimator: [Levina and Bickel, Maximum Likelihood Estimation of Intrinsic Dimension](https://papers.neurips.cc/paper/2577-maximum-likelihood-estimation-of-intrinsic-dimension.pdf). Restricted-neighbor comparisons here are a diagnostic extension of the established pipeline.